<a href="https://colab.research.google.com/github/samar45/Genai/blob/main/hipster1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
! pip install transformers datasets sacrebleu
! pip install -U evaluate



In [ ]:
import torch

import evaluate
from datasets import load_dataset
from transformers import AutoTokenizer
from transformers import EncoderDecoderModel
from transformers import Seq2SeqTrainer, Seq2SeqTrainingArguments
from transformers import MarianMTModel, Seq2SeqTrainingArguments, Seq2SeqTrainer, DataCollatorForSeq2Seq, MarianConfig

In [ ]:
from datasets import load_dataset

# Load Tatoeba dataset (English to Hindi)
dataset = load_dataset("tatoeba", lang1="en", lang2="hi")

# Inspect the dataset
print(dataset['train'])


In [ ]:
dataset['train'][:10]


In [ ]:
# Preprocess the dataset (creating 'source' and 'target' columns)
def preprocess_data(example):
    return {
        "source": example["translation"]["hi"],  # Hindi source sentence
        "target": example["translation"]["en"],  # English target sentence
    }

processed_dataset = dataset['train'].map(preprocess_data)

# Optionally, filter out long sentences (e.g., > 128 tokens)
MAX_LENGTH = 128
processed_dataset = processed_dataset.filter(
    lambda x: len(x["source"].split()) <= MAX_LENGTH and len(x["target"].split()) <= MAX_LENGTH
)

# Show a sample
print(processed_dataset[0])


In [ ]:
# Load the tokenizer for M2M100
model_name = "facebook/m2m100_418M"
tokenizer = M2M100Tokenizer.from_pretrained(model_name)

# Set source and target languages
tokenizer.src_lang = "hi"
tokenizer.tgt_lang = "en"

# Tokenization with padding and truncation for both source and target
def tokenize_function(example):
    # Tokenize the source (Hindi) with padding and truncation
    model_inputs = tokenizer(example["source"], max_length=MAX_LENGTH, padding="max_length", truncation=True)

    # For labels (target: English), ensure padding and truncation as well
    with tokenizer.as_target_tokenizer():
        labels = tokenizer(example["target"], max_length=MAX_LENGTH, padding="max_length", truncation=True)

    # Add labels to the model inputs
    model_inputs["labels"] = labels["input_ids"]
    return model_inputs

# Tokenize the dataset
tokenized_dataset = processed_dataset.map(tokenize_function, batched=True)

# Show the tokenized data
print(tokenized_dataset[0])


In [ ]:
print(tokenized_dataset[0])  # Check the first example


In [ ]:
import os
os.environ["WANDB_DISABLED"] = "true"


In [ ]:
from transformers import Seq2SeqTrainer, Seq2SeqTrainingArguments
from transformers import M2M100ForConditionalGeneration

# Load the model for fine-tuning
model = M2M100ForConditionalGeneration.from_pretrained(model_name)

# Define training arguments with simplified hyperparameters and logging
training_args = Seq2SeqTrainingArguments(
    output_dir="./results",  # Directory to save results
    evaluation_strategy="epoch",  # Evaluate after each epoch
    save_strategy="epoch",  # Save the model every epoch
    learning_rate=5e-5,  # Set learning rate
    per_device_train_batch_size=8,  # Batch size for training
    per_device_eval_batch_size=8,  # Batch size for evaluation
    num_train_epochs=3,  # Number of epochs
    weight_decay=0.01,  # Weight decay for regularization
    logging_dir='./logs',  # Directory for logs
    logging_steps=500,  # Log every 500 steps
    load_best_model_at_end=True,  # Load the best model based on evaluation
    predict_with_generate=True,  # Enable prediction during evaluation
    report_to="none",  # Disable external logging (e.g., WandB)
)

# Initialize the Trainer
trainer = Seq2SeqTrainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_dataset,
    eval_dataset=tokenized_dataset,  # Or a separate validation dataset if available
    tokenizer=tokenizer,
)

# Fine-tune the model
trainer.train()


In [ ]:
import evaluate

# Load the BLEU metric
bleu = evaluate.load("sacrebleu")

# Evaluation function
def evaluate_model(dataset, model, tokenizer):
    sources = dataset["source"]
    targets = [dataset["target"]]
    predictions = []

    for src in sources:
        inputs = tokenizer(src, return_tensors="pt", truncation=True)
        outputs = model.generate(**inputs)
        decoded = tokenizer.decode(outputs[0], skip_special_tokens=True)
        predictions.append(decoded)

    bleu_score = bleu.compute(predictions=predictions, references=targets)
    print(f"BLEU score: {bleu_score['score']}")

# Evaluate the fine-tuned model
evaluate_model(tokenized_dataset, model, tokenizer)
